# 01 - Data Cleaning

Raw file: `data/raw/amazon.csv` (Amazon India product listings with prices, ratings and customer reviews).

In this notebook I only clean and prepare the data. The analysis is done in `02_exploratory_analysis.ipynb`.

Steps:
1. Look at the raw data
2. Fix price, discount, rating and rating-count columns (they are stored as text)
3. Remove duplicate products
4. Split the category column and extract the brand
5. Create a few new columns for analysis
6. Save the clean file for Python, SQL, Excel and Power BI / Tableau

In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

raw = pd.read_csv("../data/raw/amazon.csv")
df = raw.copy()
print("Rows, columns:", df.shape)

Rows, columns: (1465, 16)


## 1. First look

In [2]:
df[["product_id", "product_name", "category", "discounted_price", "actual_price",
    "discount_percentage", "rating", "rating_count"]].head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269"
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994"
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928"
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363"
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905"


In [3]:
df.dtypes

product_id             str
product_name           str
category               str
discounted_price       str
actual_price           str
discount_percentage    str
rating                 str
rating_count           str
about_product          str
user_id                str
user_name              str
review_id              str
review_title           str
review_content         str
img_link               str
product_link           str
dtype: object

In [4]:
df.isnull().sum()

product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           2
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
dtype: int64

**Problems I can see:**
- `discounted_price`, `actual_price`, `discount_percentage`, `rating_count` are all text (they have the ₹ symbol, commas or a % sign), so I cannot do maths on them yet.
- `rating_count` has 2 missing values.
- `rating` is also text. Let me check why.

In [5]:
# which rating values are not a normal number like 4.2 ?
bad_rating = df[~df["rating"].str.match(r"^\d(\.\d)?$")]
bad_rating[["product_id", "product_name", "rating", "rating_count"]]

,product_id,product_name,rating,rating_count
1279,B08L12N5H1,Eureka Forbes car Vac 100 Watts Powerful Sucti...,|,992


One product has `|` as its rating. That is a data entry error, so I will turn it into a missing value (NaN) instead of guessing a rating.

## 2. Convert text columns to numbers

In [6]:
# remove the rupee sign and commas, then convert to number
for col in ["discounted_price", "actual_price"]:
    df[col] = pd.to_numeric(df[col].str.replace("[₹,]", "", regex=True))

# 64% -> 64
df["discount_pct"] = pd.to_numeric(df["discount_percentage"].str.rstrip("%"))

# rating: '|' becomes NaN
df["rating"] = pd.to_numeric(df["rating"].replace("|", np.nan))

# rating count: "24,269" -> 24269
df["rating_count"] = pd.to_numeric(df["rating_count"].str.replace(",", ""))

df[["discounted_price", "actual_price", "discount_pct", "rating", "rating_count"]].describe().round(2)

,discounted_price,actual_price,discount_pct,rating,rating_count
count,1465.00,1465.00,1465.00,1464.00,1463.00
mean,3125.31,5444.99,47.69,4.10,18295.54
std,6944.30,10874.83,21.64,0.29,42753.86
min,39.00,39.00,0.00,2.00,2.00
25%,325.00,800.00,32.00,4.00,1186.00
50%,799.00,1650.00,50.00,4.10,5179.00
75%,1999.00,4295.00,63.00,4.30,17336.50
max,77990.00,139900.00,94.00,5.00,426973.00


## 3. Duplicate products

In [7]:
print("Rows:", len(df))
print("Unique product_id:", df["product_id"].nunique())
print("Repeated product_id rows:", df["product_id"].duplicated().sum())
print("Fully identical rows:", df.duplicated().sum())

Rows: 1465
Unique product_id: 1351
Repeated product_id rows: 114
Fully identical rows: 0


No row is fully identical, but 114 rows repeat a `product_id`. When I checked a few of them, the same product appears twice with a tiny difference (for example the rating count is 1,07,687 in one row and 1,07,686 in the other), which means the data was scraped at slightly different times.

I will keep one row per product - the one with the highest rating count (the latest scrape).

In [8]:
df = (df.sort_values("rating_count", ascending=False)
        .drop_duplicates(subset="product_id", keep="first")
        .sort_index()
        .reset_index(drop=True))
print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1351


## 4. Missing values

After the conversion, 1 rating and 2 rating counts are missing. I am **not filling** them with mean or median, because a made-up rating would affect the results. They are only 3 rows out of about 1,350, so I leave them as NaN and ignore them wherever a calculation needs that column.

In [9]:
df[["rating", "rating_count"]].isnull().sum()

rating          1
rating_count    2
dtype: int64

## 5. Category and brand

In [10]:
print(df["category"].iloc[0])

Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables


The category is one long string separated by `|` (main category | sub category | ... | last level). I will split it into separate columns.

In [11]:
parts = df["category"].str.split("|")
df["main_category"] = parts.str[0]
df["sub_category"] = parts.str[1]
df["leaf_category"] = parts.str[-1]

df["main_category"].value_counts()

main_category
Electronics              490
Home&Kitchen             448
Computers&Accessories    375
OfficeProducts            31
MusicalInstruments         2
HomeImprovement            2
Toys&Games                 1
Car&Motorbike              1
Health&PersonalCare        1
Name: count, dtype: int64

Only 3 main categories have a decent number of products (Electronics, Home&Kitchen, Computers&Accessories). The other 6 have 31 products or fewer, so I will be careful while comparing them later.

**Brand:** there is no brand column, so I take the first word of the product name. Spelling differs (`boAt`, `Boat`, `BOAT`), so I group by lowercase name and show the most common spelling. `Amazon Basics` and `AmazonBasics` are the same brand and need a small fix.

In [12]:
first_word = df["product_name"].str.split().str[0]
brand_key = first_word.str.lower()

# "Amazon Basics ..." (two words) should match "AmazonBasics ..."
two_word_amazon = df["product_name"].str.lower().str.startswith("amazon basics")
brand_key = brand_key.mask(two_word_amazon, "amazonbasics")

# for each key, use the spelling that appears most often
display_name = (pd.DataFrame({"key": brand_key, "word": first_word})
                  .groupby("key")["word"].agg(lambda s: s.value_counts().index[0]))
df["brand"] = brand_key.map(display_name)

df["brand"].value_counts().head(10)

brand
boAt            53
AmazonBasics    48
Samsung         33
Zebronics       30
Bajaj           27
Redmi           25
MI              24
Philips         23
Portronics      21
Wayona          21
Name: count, dtype: int64

This is not perfect (for example `Mi` and `Redmi` are both Xiaomi, and a few product names start with a word that is not a brand), but it is good enough for a brand-level overview.

## 6. New columns for analysis

In [13]:
# how many rupees the customer saves
df["amount_saved"] = df["actual_price"] - df["discounted_price"]

# price bands based on the selling price
df["price_band"] = pd.cut(
    df["discounted_price"],
    bins=[0, 200, 500, 1000, 2000, 5000, np.inf],
    labels=["Under 200", "200-500", "500-1000", "1000-2000", "2000-5000", "5000+"],
)

# discount bands
df["discount_band"] = pd.cut(
    df["discount_pct"],
    bins=[-1, 20, 40, 60, 80, 100],
    labels=["0-20%", "21-40%", "41-60%", "61-80%", "81%+"],
)

# number of reviews that are actually present in the file for this product
df["n_reviews"] = df["review_id"].str.split(",").str.len()

df[["discounted_price", "actual_price", "amount_saved", "price_band", "discount_band", "n_reviews"]].head()

,discounted_price,actual_price,amount_saved,price_band,discount_band,n_reviews
0,199.0,349.0,150.0,Under 200,41-60%,8
1,154.0,399.0,245.0,Under 200,61-80%,8
2,149.0,1000.0,851.0,Under 200,81%+,8
3,229.0,299.0,70.0,200-500,21-40%,8
4,499.0,999.0,500.0,200-500,41-60%,8


### Review text signals

Each row has several reviews joined into one cell. The reviews contain commas themselves, so I cannot split them cleanly into one review per row. Instead I count simple positive and negative keywords in the combined review text of each product and divide by the number of reviews. It is a rough signal, not real sentiment analysis, but it is enough to compare categories.

I left out words that can mean both things, such as "bad" (as in "not bad"), "return" and "heating" (a heater is supposed to heat).

In [14]:
positive_words = ["good", "great", "excellent", "value for money", "durable",
                  "fast", "perfect", "love", "nice", "best", "sturdy", "recommend"]
negative_words = ["not working", "stopped working", "waste", "poor", "broke", "broken",
                  "defective", "fake", "worst", "disappoint", "damaged", "useless", "refund"]

text = (df["review_title"].fillna("") + " " + df["review_content"].fillna("")).str.lower()

def count_words(words):
    total = pd.Series(0, index=df.index)
    for w in words:
        total = total + text.str.count(re.escape(w))
    return total

df["positive_mentions"] = count_words(positive_words)
df["negative_mentions"] = count_words(negative_words)
df["positive_per_review"] = (df["positive_mentions"] / df["n_reviews"]).round(3)
df["negative_per_review"] = (df["negative_mentions"] / df["n_reviews"]).round(3)

df[["product_name", "rating", "positive_per_review", "negative_per_review"]].head()

,product_name,rating,positive_per_review,negative_per_review
0,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,4.0,2.125,0.000
1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,4.2,2.250,0.375
2,pTron Solero TB301 3A Type-C Data and Fast Cha...,3.9,1.625,0.000
3,MI Usb Type-C Cable Smartphone (Black),4.3,1.125,0.000
4,"TP-Link USB WiFi Adapter for PC(TL-WN725N), N1...",4.2,3.000,0.250


## 7. Sanity checks

In [15]:
# 1. discounted price should never be more than actual price
print("Discounted > actual:", (df["discounted_price"] > df["actual_price"]).sum())

# 2. does the discount % in the file match the prices?
calc_pct = ((df["actual_price"] - df["discounted_price"]) / df["actual_price"] * 100).round()
print("Discount % mismatches (more than 1 point):", ((calc_pct - df["discount_pct"]).abs() > 1).sum())

# 3. ratings should be between 1 and 5
print("Rating range:", df["rating"].min(), "to", df["rating"].max())

# 4. product_id is unique now
print("product_id unique:", df["product_id"].is_unique)

Discounted > actual: 0
Discount % mismatches (more than 1 point): 0
Rating range: 2.0 to 5.0
product_id unique: True


All checks pass. The discount % given in the file is consistent with the two prices.

One more thing I noticed: in some rows the `review_content` contains image links (`https://m.media-amazon.com/...`) instead of text. I leave them in because they are only a small part of the text and do not change the keyword counts in any meaningful way.

## 8. Save clean data

In [16]:
keep = ["product_id", "product_name", "brand", "main_category", "sub_category", "leaf_category",
        "actual_price", "discounted_price", "discount_pct", "amount_saved",
        "price_band", "discount_band", "rating", "rating_count", "n_reviews",
        "positive_per_review", "negative_per_review", "product_link"]

clean = df[keep]
clean.to_csv("../data/processed/amazon_products_clean.csv", index=False)
print("Saved:", clean.shape)
clean.head()

Saved: (1351, 18)


,product_id,product_name,brand,main_category,sub_category,leaf_category,actual_price,discounted_price,discount_pct,amount_saved,price_band,discount_band,rating,rating_count,n_reviews,positive_per_review,negative_per_review,product_link
0,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Ambrane,Computers&Accessories,Accessories&Peripherals,USBCables,349.0,199.0,43,150.0,Under 200,41-60%,4.0,43994.0,8,2.125,0.000,https://www.amazon.in/Ambrane-Unbreakable-Char...
1,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Portronics,Computers&Accessories,Accessories&Peripherals,USBCables,399.0,154.0,61,245.0,Under 200,61-80%,4.2,16905.0,8,2.250,0.375,https://www.amazon.in/Portronics-Konnect-POR-1...
2,B08Y1TFSP6,pTron Solero TB301 3A Type-C Data and Fast Cha...,pTron,Computers&Accessories,Accessories&Peripherals,USBCables,1000.0,149.0,85,851.0,Under 200,81%+,3.9,24871.0,8,1.625,0.000,https://www.amazon.in/Solero-TB301-Charging-48...
3,B08DDRGWTJ,MI Usb Type-C Cable Smartphone (Black),MI,Computers&Accessories,Accessories&Peripherals,USBCables,299.0,229.0,23,70.0,200-500,21-40%,4.3,30411.0,8,1.125,0.000,https://www.amazon.in/MI-MTCY001IN-USB-Type-C-...
4,B008IFXQFU,"TP-Link USB WiFi Adapter for PC(TL-WN725N), N1...",TP-Link,Computers&Accessories,NetworkingDevices,WirelessUSBAdapters,999.0,499.0,50,500.0,200-500,41-60%,4.2,179691.0,8,3.000,0.250,https://www.amazon.in/TP-Link-TL-WN725N-150Mbp...
